In [2]:
# ── Cell 1: Environment verification ─────────────────────────────────────────
# Project: MV Feeder Protection Study
# Tool:    pandapower + Python
# Region:  South Africa / SADC — Eskom-style 11 kV radial feeder
# Author:  Hillary M
# ─────────────────────────────────────────────────────────────────────────────

import pandapower as pp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Verify pandapower version
print(f"pandapower version: {pp.__version__}")

# Confirm installation is functional
net = pp.create_empty_network()
print("\nEmpty network created successfully:")
print(net)

pandapower version: 3.4.0

Empty network created successfully:
This pandapower network is empty


In [3]:
# ── Cell 2: Network Topology Definition ──────────────────────────────────────
# Eskom-style 11 kV radial feeder — Mpumalanga coalfields context
# Topology:
# Bus0(132kV) → [Trafo] → Bus1(11kV) → [CB/R1] → Line1→2
#             → Bus2(11kV) → [CB/R2] → Line2→3
#             → Bus3(11kV) → [CB/R3] → Mine load
# ─────────────────────────────────────────────────────────────────────────────

net = pp.create_empty_network(name="Mpumalanga 11kV Radial Feeder", f_hz=50, sn_mva=1)

# ── Buses ─────────────────────────────────────────────────────────────────────
bus0 = pp.create_bus(net, vn_kv=132, name="Bus 0 - Grid Infeed (132kV)")
bus1 = pp.create_bus(net, vn_kv=11,  name="Bus 1 - Zone 1 Substation (11kV)")
bus2 = pp.create_bus(net, vn_kv=11,  name="Bus 2 - Zone 2 Mid-Feeder (11kV)")
bus3 = pp.create_bus(net, vn_kv=11,  name="Bus 3 - Mine Incomer (11kV)")

print(f"Buses created: {len(net.bus)}")
print(net.bus[['name', 'vn_kv']])

# ── External grid — Eskom 132 kV swing bus ────────────────────────────────────
# s_sc_max_mva: max fault level (strong grid, peak generation) = 250 MVA
# s_sc_min_mva: min fault level (weak grid, light generation)  = 120 MVA
# rx_max/rx_min: R/X ratio of source = 0.1 (source is mostly inductive)
pp.create_ext_grid(
    net,
    bus=bus0,
    vm_pu=1.0,
    va_degree=0,
    s_sc_max_mva=250,
    s_sc_min_mva=120,
    rx_max=0.1,
    rx_min=0.1,
    name="Eskom Grid Infeed"
)

print(f"\nExternal grid created: {len(net.ext_grid)}")

# ── Transformer: 132/11 kV, 40 MVA ───────────────────────────────────────────
# vk_percent  = 10%   → from assignment brief
# vkr_percent = 0.5%  → derived: X/R=20 from brief, so R = vk/20 = 10/20 = 0.5%
# pfe_kw      = 30 kW → standard for 40 MVA Eskom distribution transformer
# i0_percent  = 0.05% → standard no-load current for 40 MVA ONAN transformer
pp.create_transformer_from_parameters(
    net,
    hv_bus=bus0,
    lv_bus=bus1,
    sn_mva=40,
    vn_hv_kv=132,
    vn_lv_kv=11,
    vkr_percent=0.5,
    vk_percent=10.0,
    pfe_kw=30,
    i0_percent=0.05,
    name="132/11kV 40MVA Transformer"
)

print(f"\nTransformer created: {len(net.trafo)}")

# ── Line 1→2: 9 km Pelican ACSR overhead line ────────────────────────────────
# This line is LINE INDEX 0 in net.line
# r_ohm_per_km = 0.306  → resistance per km of Pelican conductor
# x_ohm_per_km = 0.383  → reactance per km of Pelican conductor
# c_nf_per_km  = 8.0    → capacitance (typical ACSR overhead line)
# max_i_ka     = 0.173  → thermal limit: 3.3MVA / (√3 × 11kV) = 0.173 kA
pp.create_line_from_parameters(
    net,
    from_bus=bus1,
    to_bus=bus2,
    length_km=9,
    r_ohm_per_km=0.306,
    x_ohm_per_km=0.383,
    c_nf_per_km=8.0,
    max_i_ka=0.173,
    name="Line 1-2 (9km Pelican ACSR)"
)

# ── Line 2→3: 6 km Pelican ACSR overhead line ────────────────────────────────
# This line is LINE INDEX 1 in net.line
pp.create_line_from_parameters(
    net,
    from_bus=bus2,
    to_bus=bus3,
    length_km=6,
    r_ohm_per_km=0.306,
    x_ohm_per_km=0.383,
    c_nf_per_km=8.0,
    max_i_ka=0.173,
    name="Line 2-3 (6km Pelican ACSR)"
)

print(f"\nLines created: {len(net.line)}")
print(net.line[['name', 'length_km', 'r_ohm_per_km', 'x_ohm_per_km', 'max_i_ka']])

# ── Loads ─────────────────────────────────────────────────────────────────────
pp.create_load(net, bus=bus2, p_mw=0.4,  q_mvar=0.2, name="Rural Load (Bus 2)")
pp.create_load(net, bus=bus3, p_mw=1.2,  q_mvar=0.6, name="Mine Load (Bus 3)")

print(f"\nLoads created: {len(net.load)}")
print(net.load[['name', 'bus', 'p_mw', 'q_mvar']])

# ── Circuit breakers ──────────────────────────────────────────────────────────
# Each switch needs two things to place it correctly:
#   bus     = which bus the CB sits at
#   element = which LINE it is on (line index in net.line)
#
# sw1: Bus 1, LINE 0 (Line 1→2) — CB at sending end of Line 1→2, relay R1
# sw2: Bus 2, LINE 1 (Line 2→3) — CB at sending end of Line 2→3, relay R2
# sw3: Bus 3, LINE 1 (Line 2→3) — CB at receiving end of Line 2→3, relay R3
#      sw2 and sw3 both reference element=1 because BOTH sit on Line 2→3.
#      The bus parameter is what places them at opposite ends of that line.

sw1 = pp.create_switch(net, bus=bus1, element=0, et='l', closed=True, name="CB R1 (Bus1 - Line1-2 sending end)")
sw2 = pp.create_switch(net, bus=bus2, element=1, et='l', closed=True, name="CB R2 (Bus2 - Line2-3 sending end)")
sw3 = pp.create_switch(net, bus=bus3, element=1, et='l', closed=True, name="CB R3 (Bus3 - Line2-3 receiving end)")

print(f"\nSwitches created: {len(net.switch)}")
print(net.switch[['name', 'bus', 'element', 'et', 'closed']])

# ── Final network summary ─────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("FINAL NETWORK SUMMARY")
print("=" * 55)
print(f"Buses:        {len(net.bus)}   (Bus 0–3)")
print(f"Ext grids:    {len(net.ext_grid)}   (Eskom 132kV swing)")
print(f"Transformers: {len(net.trafo)}   (132/11kV 40MVA)")
print(f"Lines:        {len(net.line)}   (Line 1→2: 9km, Line 2→3: 6km)")
print(f"Loads:        {len(net.load)}   (Rural 400kW, Mine 1.2MW)")
print(f"Switches:     {len(net.switch)}   (CB R1 @ Bus1, CB R2 @ Bus2, CB R3 @ Bus3)")

Buses created: 4
                               name  vn_kv
0       Bus 0 - Grid Infeed (132kV)  132.0
1  Bus 1 - Zone 1 Substation (11kV)   11.0
2  Bus 2 - Zone 2 Mid-Feeder (11kV)   11.0
3       Bus 3 - Mine Incomer (11kV)   11.0

External grid created: 1

Transformer created: 1

Lines created: 2
                          name  length_km  r_ohm_per_km  x_ohm_per_km  \
0  Line 1-2 (9km Pelican ACSR)        9.0         0.306         0.383   
1  Line 2-3 (6km Pelican ACSR)        6.0         0.306         0.383   

   max_i_ka  
0     0.173  
1     0.173  

Loads created: 2
                 name  bus  p_mw  q_mvar
0  Rural Load (Bus 2)    2   0.4     0.2
1   Mine Load (Bus 3)    3   1.2     0.6

Switches created: 3
                                   name  bus  element et  closed
0    CB R1 (Bus1 - Line1-2 sending end)    1        0  l    True
1    CB R2 (Bus2 - Line2-3 sending end)    2        1  l    True
2  CB R3 (Bus3 - Line2-3 receiving end)    3        1  l    True

FINAL NETWORK S

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


BUS RESULTS

Bus    Name                                V (pu)     V (kV)     Status
---------------------------------------------------------------------------
0      Bus 0 - Grid Infeed (132kV)         1.0000     132.000    OK
1      Bus 1 - Zone 1 Substation (11kV)    0.9974     10.971     OK
2      Bus 2 - Zone 2 Mid-Feeder (11kV)    0.9314     10.245     OK
3      Bus 3 - Mine Incomer (11kV)         0.8983     9.881      VIOLATION


LINE RESULTS

Line   Name                           I (kA)     I_max (kA)   Loading %    Status
--------------------------------------------------------------------------------
0      Line 1-2 (9km Pelican ACSR)    0.1036     0.1730       59.9         OK
1      Line 2-3 (6km Pelican ACSR)    0.0784     0.1730       45.3         OK


TRANSFORMER RESULTS

Trafo  Name                           P_HV (MW)    Q_HV (MVAr)    Loading %    Status
--------------------------------------------------------------------------------
0      132/11kV 40MVA Transformer  